# Hugging Face Prompt Filtering

In [ ]:
import os
import sys
sys.path.insert(0, ".")

from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from src.common.data import create_fixed_splits, load_bnsentmix
from src.common.io_utils import batch_sentences, append_jsonl
from src.filtering.prompts import decisions_path
from src.filtering.prompts import build_batch_prompt, parse_batch_response, prompt_version_label
PROMPT_VERSION = "v3"

load_dotenv(override=True)
HF_TOKEN = os.environ["HF_TOKEN"]
MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"
hf_client = InferenceClient(api_key=HF_TOKEN)
print("Hugging Face client configured.")

# Test One Shared Batch

In [ ]:
splits = create_fixed_splits(load_bnsentmix())
batch = next(batch_sentences(splits["train"], batch_size=3))
response = hf_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": build_batch_prompt(batch, PROMPT_VERSION)}],
    temperature=0,
    max_tokens=512,
)
decisions = parse_batch_response(response.choices[0].message.content, len(batch))
print(decisions)

# Configure a Resumable Full Run

In [ ]:
import json
BATCH_SIZE = 50
DECISIONS_PATH = decisions_path("huggingface", MODEL_NAME, PROMPT_VERSION)
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
completed_indices = set()
if DECISIONS_PATH.exists():
    with DECISIONS_PATH.open(encoding="utf-8") as file:
        for line in file:
            if line.strip():
                completed_indices.add(json.loads(line)["sentence_index"])
print("Result file:", DECISIONS_PATH)
print("Already completed:", len(completed_indices))

In [ ]:
for start in range(0, len(splits["train"]), BATCH_SIZE):
    stop = min(start + BATCH_SIZE, len(splits["train"]))
    indices = [index for index in range(start, stop) if index not in completed_indices]
    if not indices:
        continue
    sentences = [splits["train"][index]["Sentence"] for index in indices]
    response = hf_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": build_batch_prompt(sentences, PROMPT_VERSION)}],
        temperature=0,
        max_tokens=512,
    )
    decisions = parse_batch_response(response.choices[0].message.content, len(sentences))
    records = [
        {"sentence_index": index, "keep": keep, "provider": "huggingface", "model": MODEL_NAME, "prompt_version": prompt_version_label(PROMPT_VERSION)}
        for index, keep in zip(indices, decisions)
    ]
    append_jsonl(records, DECISIONS_PATH)
    completed_indices.update(indices)
    print(f"Completed {len(completed_indices)}/{len(splits['train'])}")